# Демо 1. Класичне донавчання: зміна голови в моделі BERT

У цьому ноутбуці донавчаємо модель **BERT** для задачі **визначення парафраз**: модель отримує два речення й визначає, чи передають вони той самий зміст.

План:
1. Завантажити датасет MRPC і токенізувати пари речень.
2. Завантажити BERT із **новою головою**-класифікатором.
3. **Повне донавчання** з `Trainer`: тренуються всі ваги (кістяк і голова).
4. Перевірити модель на власних прикладах.

Приклад за мотивами розділу 3 курсу [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter3/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [1]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [2]:
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Пристрій: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("Включіть GPU, бо набридне чекати ;)")

Пристрій: cuda
GPU: Tesla T4


## 1. Дані: MRPC

**MRPC** (Microsoft Research Paraphrase Corpus) — пари речень з новинних статей.
Мітка `1` означає, що речення є парафразами, мітка `0` — що ні.

In [3]:
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  649kB            

mrpc/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 75.7kB            

mrpc/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  308kB            

mrpc/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [4]:
print("Приклад нееквівалентних речень:")
example = raw_datasets["train"][3]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

print("")
print("Приклад еквівалентних речень:")
example = raw_datasets["train"][5]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

Приклад нееквівалентних речень:
Речення 1: Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
Речення 2: Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
Мітка:    0 → not_equivalent

Приклад еквівалентних речень:
Речення 1: Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .
Речення 2: With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .
Мітка:    1 → equivalent


### Знайомимось з BERT BASE UNCASED

Для цього прочитайте інформацію на [картці моделі](https://huggingface.co/google-bert/bert-base-uncased)

In [5]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model='bert-base-uncased')
unmasker("The man worked as a [MASK].")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[{'score': 0.09747552871704102,
  'token': 10533,
  'token_str': 'carpenter',
  'sequence': 'the man worked as a carpenter.'},
 {'score': 0.052383244037628174,
  'token': 15610,
  'token_str': 'waiter',
  'sequence': 'the man worked as a waiter.'},
 {'score': 0.049627017229795456,
  'token': 13362,
  'token_str': 'barber',
  'sequence': 'the man worked as a barber.'},
 {'score': 0.037886038422584534,
  'token': 15893,
  'token_str': 'mechanic',
  'sequence': 'the man worked as a mechanic.'},
 {'score': 0.037680864334106445,
  'token': 18968,
  'token_str': 'salesman',
  'sequence': 'the man worked as a salesman.'}]

In [6]:
print("Не по темі, але погляньте, яка \"несправедливість\"...")
unmasker("The woman worked as a [MASK].")

Не по темі, але погляньте, яка "несправедливість"...


[{'score': 0.21981368958950043,
  'token': 6821,
  'token_str': 'nurse',
  'sequence': 'the woman worked as a nurse.'},
 {'score': 0.15973998606204987,
  'token': 13877,
  'token_str': 'waitress',
  'sequence': 'the woman worked as a waitress.'},
 {'score': 0.11547233909368515,
  'token': 10850,
  'token_str': 'maid',
  'sequence': 'the woman worked as a maid.'},
 {'score': 0.03796861320734024,
  'token': 19215,
  'token_str': 'prostitute',
  'sequence': 'the woman worked as a prostitute.'},
 {'score': 0.030423680320382118,
  'token': 5660,
  'token_str': 'cook',
  'sequence': 'the woman worked as a cook.'}]

### Токенізація пари речень

BERT приймає обидва речення як одну послідовність: `[CLS] речення 1 [SEP] речення 2 [SEP]`.
Поле `token_type_ids` позначає, до якого речення належить кожен токен (0 або 1).

Примітка: класифікаційний токен (`[CLS]`) використовується як вхідне представлення для задач класифікації в NLP та машинному навчанні. Він акумулює інформацію з усієї вхідної послідовності та передає її крізь шари моделі для подальшої обробки. Потім модель використовує це представлення для формування прогнозів або віднесення вхідних даних до заздалегідь визначених категорій.

In [7]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
encoded = tokenizer(example["sentence1"], example["sentence2"])

print("")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[:20])
print(encoded["token_type_ids"][:20])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[-20:])
print(encoded["token_type_ids"][-20:])


['[CLS]', 'revenue', 'in', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[',', 'revenue', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.', '[SEP]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [8]:
def tokenize_function(batch):
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

## 2. Модель: кістяк + нова голова

`AutoModelForSequenceClassification` бере кістяк (backbone) BERT і додає нову голову-класифікатор
з `num_labels` виходами. Зверніть увагу на звіт під час завантаження:
- ваги `cls.*` позначені як не використані (`UNEXPECTED`, у старих версіях — «not used»): це відкинута голова попереднього тренування;
- ваги `classifier.*` позначені як відсутні (`MISSING`, у старих версіях — «newly initialized»): це нова голова, ініціалізована випадково.

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

print(model.classifier)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Linear(in_features=768, out_features=2, bias=True)


In [10]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


print(f"Кістяк (model.bert):   {count_params(model.bert):>12,} параметрів")
print(f"Голова (model.classifier): {count_params(model.classifier):>8,} параметрів")

Кістяк (model.bert):    109,482,240 параметрів
Голова (model.classifier):    1,538 параметрів


## 3. Повне донавчання з `Trainer`

`Trainer` — клас бібліотеки Transformers, який містить увесь цикл тренування й оцінювання.
Метрики рахуємо самі у функції `compute_metrics`.

Оновлюються **всі** ваги: і кістяк, і нова голова.
Для повного донавчання використовують малий learning rate (`2e-5`),
щоб не зруйнувати знання, отримані під час попереднього тренування.

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="bert-mrpc",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [12]:
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
trainer.train()
trainer.evaluate()

Параметрів, що тренуються: 109,483,778


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.487028,0.381455,0.838235,0.886986
2,0.328676,0.357605,0.848039,0.893471
3,0.167169,0.416541,0.860294,0.901554


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.167169,0.416541,3,0.860294,0.901554


{'eval_loss': 0.4165409803390503,
 'eval_accuracy': 0.8602941176470589,
 'eval_f1': 0.9015544041450777}

## 4. Перевірка на власних прикладах

In [13]:
def predict(model, sentence1, sentence2):
    model.eval()
    inputs = tokenizer(sentence1, sentence2, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return f"парафраз: {probs[1]:.1%}"


pairs = [
    ("The company reported a profit of $5 million.", "The firm earned five million dollars in profit."),
    ("The company reported a profit of $5 million.", "The weather was sunny all week."),
    ("She bought a new car yesterday.", "Yesterday she purchased a new vehicle."),
    # 4 додаткові перевірки на власних даних (пункт 10):
    ("Scientists discovered a new planet outside our solar system.", "A new exoplanet beyond our solar system was found by researchers."),
    ("The flight was delayed by three hours due to heavy fog.", "The flight arrived three hours early despite clear skies."),
    ("More than half of the students passed the final exam.", "Over 50% of the learners succeeded in the last test."),
    ("Apple announced a new iPhone model in September.", "Samsung released a new Galaxy smartphone in October."),
]
for s1, s2 in pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The company reported a profit of $5 million.
The firm earned five million dollars in profit.
→ парафраз: 86.0%

The company reported a profit of $5 million.
The weather was sunny all week.
→ парафраз: 2.6%

She bought a new car yesterday.
Yesterday she purchased a new vehicle.
→ парафраз: 99.0%

Scientists discovered a new planet outside our solar system.
A new exoplanet beyond our solar system was found by researchers.
→ парафраз: 98.8%

The flight was delayed by three hours due to heavy fog.
The flight arrived three hours early despite clear skies.
→ парафраз: 78.0%

More than half of the students passed the final exam.
Over 50% of the learners succeeded in the last test.
→ парафраз: 55.4%

Apple announced a new iPhone model in September.
Samsung released a new Galaxy smartphone in October.
→ парафраз: 98.1%



## 5. Відповіді на теоретичні питання (пункти 1–10)

### 1. Як довго виконувалися клітинки ноутбуку `01_bert_head`? Чи були проблеми з запуском, які?

На середовищі Google Colab з відеокартою Tesla T4 GPU імпорт бібліотек зайняв 39 секунд, завантаження датасету MRPC, ваг моделі `bert-base-uncased` (~440 МБ) та токенізація - 12 секунд, а повне тренування за 3 епохи у змішаній точності `fp16` (690 кроків) - 1 хвилину 4 секунди (сумарний час виконання всіх клітинок разом із додатковим експериментом на 8 епохах у пункті 9 - близько 4.5 хвилин).

Проблем із запуском не виникло. Попередження під час завантаження `AutoModelForSequenceClassification` про невикористані ваги `cls.*` (`UNEXPECTED`) та новоініціалізовану голову `classifier.*` (`MISSING`) є очікуваними й підтверджують правильну заміну голови.

### 2. Як Ви розумієте, що таке токен `[CLS]`? Яким чином він виконує свою функцію?

Токен **`[CLS]`** (classification) - це спеціальний службовий токен, який токенізатор BERT завжди додає на першу позицію вхідної послідовності (`[CLS] речення 1 [SEP] речення 2 [SEP]`).

Завдяки механізму двонапрямленої самоуваги (bidirectional self-attention) у всіх 12 шарах енкодера BERT кожен токен обмінюється інформацією з усіма іншими токенами. Оскільки `[CLS]` не прив'язаний до конкретного слова тексту, його прихований вектор на виході останнього шару акумулює інтегральне семантичне представлення всієї пари речень. Саме цей вектор (розмірності 768) передається в голову класифікації `model.classifier` для прийняття рішення.

### 3. Яке пряме застосування базової моделі `bert-base-uncased`?

`bert-base-uncased` - це попередньо натренований енкодер на основі архітектури Transformer (без урахування регістру символів), навчений на корпусах BookCorpus та English Wikipedia на двох задачах: Masked Language Modeling (MLM) (передбачення замаскованих токенів `[MASK]`) та Next Sentence Prediction (NSP).

Без донавчання модель застосовується для заповнення пропусків у тексті (`fill-mask`) або побудови контекстних векторних ембедінгів слів і речень. Проте її основне практичне призначення - слугувати універсальним **кістяком (backbone)** для подальшого донавчання на прикладних задачах аналізу та класифікації тексту (визначення парафраз, тональності відгуків, виявлення спаму, NER, пошук відповідей у тексті).

### 4. Як Ви вважаєте, чи варто боротися (і якщо так, то яким чином) з проявами сексизму LLM-моделей?

Так, варто, особливо коли мовні моделі інтегруються в прикладні системи прийняття рішень (рекрутинг, скоринг, освіта, клієнтська підтримка), де відтворення гендерних чи інших соціальних стереотипів із тренувальних веб-текстів може призводити до несправедливої дискримінації.

Основні способи боротьби:
  1. Фільтрація токсичних корпусів і використання аугментації зі зміною демографічних маркерів (Counterfactual Data Augmentation).
  2. Вирівнювання (SFT, RLHF, DPO): тренування на парах відповідей, де нейтральні, об'єктивні й неупереджені формулювання отримують вищу оцінку.
  3. Явне задання вимог нейтральності в `system`-промпті та автоматична перевірка згенерованих відповідей класифікаторами безпеки.

### 5. Для чого в секції «Токенізація пари речень» виводяться числа 0 і 1?

Це значення масиву **`token_type_ids`** (segment embeddings). Оскільки BERT отримує два речення як єдиний рядок `[CLS] речення 1 [SEP] речення 2 [SEP]`, моделі необхідно розрізняти, до якого саме речення належить кожен токен.

Число **`0`** позначає токени першого речення (разом із `[CLS]` та першим `[SEP]`), а число **`1`** - токени другого речення (разом із завершальним `[SEP]`).

### 6. Що позначають числа 768 і 2, що виводяться після завантаження `AutoModelForSequenceClassification.from_pretrained()`?

У виводі `Linear(in_features=768, out_features=2, bias=True)`:

**`768`** - це розмірність прихованого вектора (`hidden_size`) моделі `bert-base-uncased`, тобто розмір вихідного вектора токена `[CLS]`, який подається з кістяка на вхід класифікатора;

**`2`** — це кількість вихідних нейронів (`num_labels=2`), що відповідає двом класам задачі MRPC: `0` (не парафраз) та `1` (парафраз).

### 7. Те що в кістяку (backbone) набагато більше параметрів ніж в голові, це добре чи погано в нашому випадку?

У кістяку міститься **109 482 240** параметрів, а в новій голові - лише **1 538** ($768 \times 2 + 2$). У нашому випадку це добре:

Уся складна мовна база (граматика, семантика, синонімія, зв'язки між словами) уже закладена в 109.5 млн параметрів кістяка під час попереднього тренування.

Компактна голова лише виконує лінійну проєкцію готового семантичного представлення у 2 класи. Це дозволяє моделі швидко навчатися на відносно невеликому датасеті MRPC (3 668 прикладів) без миттєвого перенавчання.

### 8. Що саме вимірюється за допомогою метрик accuracy & F1 в `01_bert_head`?

Метрики оцінюють якість бінарної класифікації пар речень на валідаційній вибірці MRPC:

Accuracy - загальна частка правильно класифікованих пар речень серед усіх прикладів;

F1-score - середнє гармонійне між точністю (*Precision*, часткою справжніх парафразів серед усіх, які модель позначила як `1`) та повнотою (Recall, часткою знайдених моделлю реальних парафразів).

Поєднання F1 з Accuracy важливе через дисбаланс класів у MRPC (~68% позитивних прикладів проти ~32% негативних): F1 гарантує, що модель не просто часто вгадує домінуючий клас, а коректно розрізняє обидва класи.

### 9. Експеримент зі збільшенням кількості епох тренування (`num_train_epochs=8`)
Нижче запущено тренування нової копії моделі на 8 епохах замість 3, щоб простежити динаміку `Training Loss`, `Validation Loss`, `Accuracy` та `F1`.

In [14]:
# Пункт 9: треную свіжу модель на 8 епохах і фіксую історію метрик
set_seed(42)
model_8ep = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

training_args_8ep = TrainingArguments(
    output_dir="bert-mrpc-8ep",
    num_train_epochs=8,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

trainer_8ep = Trainer(
    model=model_8ep,
    args=training_args_8ep,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_8ep.train()

# Вивожу зведену таблицю по епохах
print("\nПідсумок по епохах:")
for row in trainer_8ep.state.log_history:
    if "eval_loss" in row:
        print(f"Epoch {row['epoch']:.0f}: val_loss={row['eval_loss']:.4f}, accuracy={row['eval_accuracy']:.4f}, f1={row['eval_f1']:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.528724,0.377195,0.835784,0.880995
2,0.306423,0.353929,0.850490,0.896435
3,0.146688,0.524629,0.855392,0.890943
4,0.070833,0.643009,0.862745,0.902778
5,0.042392,0.713369,0.860294,0.902564
6,0.015914,0.816343,0.862745,0.904437
7,0.016553,0.805333,0.865196,0.905983
8,0.009529,0.784189,0.862745,0.903114



Підсумок по епохах:
Epoch 1: val_loss=0.3772, accuracy=0.8358, f1=0.8810
Epoch 2: val_loss=0.3539, accuracy=0.8505, f1=0.8964
Epoch 3: val_loss=0.5246, accuracy=0.8554, f1=0.8909
Epoch 4: val_loss=0.6430, accuracy=0.8627, f1=0.9028
Epoch 5: val_loss=0.7134, accuracy=0.8603, f1=0.9026
Epoch 6: val_loss=0.8163, accuracy=0.8627, f1=0.9044
Epoch 7: val_loss=0.8053, accuracy=0.8652, f1=0.9060
Epoch 8: val_loss=0.7842, accuracy=0.8627, f1=0.9031


Пояснення результатів пункту 9:

При збільшенні кількості епох з 3 до 8 `Training Loss` продовжує монотонно падати (до `0.0095`, тобто `< 0.01`), оскільки модель практично повністю запам'ятовує тренувальну вибірку.

Водночас `Validation Loss` досягає мінімуму на 2-й епосі (`0.3539`), після чого починає зростати (до `0.64–0.82`), а `Accuracy` і `F1` після 3–4 епох виходять на плато (`86.0–86.5%` та `90.3–90.6%` відповідно).

Причина: класичне перенавчання (overfitting). Датасет MRPC має невеликий обсяг (3 668 прикладів), а під час повного донавчання оновлюються всі 109.5 млн параметрів BERT. За 2–3 епохи модель узагальнює закономірності задачі, а далі починає підлаштовуватися під шум і конкретні формулювання тренувальних прикладів, втрачаючи здатність до узагальнення на нових даних.

### 10. Перевірка на 4 додаткових власних прикладах

Додано в секцію 4 (клітинка з `pairs`) 2 пари семантичних парафразів і 2 нееквівалентні пари зі схожою структурою речень:

1. Парафраз зі зміною активного стану на пасивний ("Scientists discovered a new planet..." / "A new exoplanet ... was found by researchers."): модель упевнено класифікує пару як парафраз (**`98.8%`**).
2. Парафраз із заміною кількісного виразу на відсотки та синонімізацією ("More than half of the students passed the final exam." / "Over 50% of the learners succeeded in the last test."): модель визначає пару як парафраз, але впевненість близька до порогу (**`55.4%`**), оскільки майже всі слова замінено синонімами з мінімальним лексичним перетином.
3. Нееквівалентні речення з протилежним змістом ("The flight was delayed by three hours due to heavy fog." / "The flight arrived three hours early despite clear skies."): модель дає хибнопозитивну оцінку (**`78.6%`**).
4. Нееквівалентні речення з різними сутностями та датами ("Apple announced a new iPhone model in September." / "Samsung released a new Galaxy smartphone in October."): модель помилково видає **`98.1%`**.

Висновок: модель BERT, донавчена на новинному корпусі MRPC, добре вловлює граматичні трансформації за наявності спільних ключових слів, але схильна до хибнопозитивних спрацювань (false positive), коли два різні за змістом речення мають однаковий синтаксичний шаблон новинної фрази (підміна власних назв, дат або антонімів у схожій конструкції).